In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EmployeeETLProject") \
    .master("local[*]") \
    .getOrCreate()

In [ ]:
from pyspark.sql import functions as F

In [ ]:
employee_data = [
    (101, "Kalyani", "IT", 50000, "Hyderabad"),
    (102, "Divya", "HR", 40000, "Hyderabad"),
    (103, "Rahul", "IT", 65000, "Bangalore"),
    (104, "Ramani", "Finance", 55000, "Hyderabad"),
    (105, "Anil", "HR", 45000, "Chennai"),
    (106, "Kiran", "IT", 70000, "Bangalore"),
    (107, "Suresh", "Finance", 60000, "Chennai"),
    (108, "Priya", "IT", 52000, "Hyderabad"),
    (109, "John", "Sales", 48000, None)
]

employee_df = spark.createDataFrame(
    employee_data,
    ["emp_id", "name", "department", "salary", "city"]
)

employee_df.show()

+------+-------+----------+------+---------+
|emp_id|   name|department|salary|     city|
+------+-------+----------+------+---------+
|   101|Kalyani|        IT| 50000|Hyderabad|
|   102|  Divya|        HR| 40000|Hyderabad|
|   103|  Rahul|        IT| 65000|Bangalore|
|   104| Ramani|   Finance| 55000|Hyderabad|
|   105|   Anil|        HR| 45000|  Chennai|
|   106|  Kiran|        IT| 70000|Bangalore|
|   107| Suresh|   Finance| 60000|  Chennai|
|   108|  Priya|        IT| 52000|Hyderabad|
|   109|   John|     Sales| 48000|     NULL|
+------+-------+----------+------+---------+



In [ ]:
employee_df.printSchema()
employee_df.show()
employee_df.count()

root
 |-- emp_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- city: string (nullable = true)

+------+-------+----------+------+---------+
|emp_id|   name|department|salary|     city|
+------+-------+----------+------+---------+
|   101|Kalyani|        IT| 50000|Hyderabad|
|   102|  Divya|        HR| 40000|Hyderabad|
|   103|  Rahul|        IT| 65000|Bangalore|
|   104| Ramani|   Finance| 55000|Hyderabad|
|   105|   Anil|        HR| 45000|  Chennai|
|   106|  Kiran|        IT| 70000|Bangalore|
|   107| Suresh|   Finance| 60000|  Chennai|
|   108|  Priya|        IT| 52000|Hyderabad|
|   109|   John|     Sales| 48000|     NULL|
+------+-------+----------+------+---------+



9

In [ ]:
employee_df.filter(
    F.col("city").isNull()
).show()

+------+----+----------+------+----+
|emp_id|name|department|salary|city|
+------+----+----------+------+----+
|   109|John|     Sales| 48000|NULL|
+------+----+----------+------+----+



In [ ]:
clean_employee_df = employee_df.fillna(
    {"city": "Unknown"}
)

clean_employee_df.show()

+------+-------+----------+------+---------+
|emp_id|   name|department|salary|     city|
+------+-------+----------+------+---------+
|   101|Kalyani|        IT| 50000|Hyderabad|
|   102|  Divya|        HR| 40000|Hyderabad|
|   103|  Rahul|        IT| 65000|Bangalore|
|   104| Ramani|   Finance| 55000|Hyderabad|
|   105|   Anil|        HR| 45000|  Chennai|
|   106|  Kiran|        IT| 70000|Bangalore|
|   107| Suresh|   Finance| 60000|  Chennai|
|   108|  Priya|        IT| 52000|Hyderabad|
|   109|   John|     Sales| 48000|  Unknown|
+------+-------+----------+------+---------+



In [ ]:
duplicate_row_df = clean_employee_df.filter(
    F.col("emp_id") == 101
)

employee_with_duplicate_df = clean_employee_df.unionByName(
    duplicate_row_df
)

print("Before removing duplicate:", employee_with_duplicate_df.count())

Before removing duplicate: 10


In [ ]:
clean_employee_df = employee_with_duplicate_df.dropDuplicates(
    ["emp_id"]
)

print("After removing duplicate:", clean_employee_df.count())

After removing duplicate: 9


In [ ]:
filtered_employee_df = clean_employee_df.filter(
    F.col("salary") >= 50000
)

filtered_employee_df.show()

+------+-------+----------+------+---------+
|emp_id|   name|department|salary|     city|
+------+-------+----------+------+---------+
|   101|Kalyani|        IT| 50000|Hyderabad|
|   103|  Rahul|        IT| 65000|Bangalore|
|   104| Ramani|   Finance| 55000|Hyderabad|
|   106|  Kiran|        IT| 70000|Bangalore|
|   107| Suresh|   Finance| 60000|  Chennai|
|   108|  Priya|        IT| 52000|Hyderabad|
+------+-------+----------+------+---------+



In [ ]:
selected_employee_df = filtered_employee_df.select(
    "name",
    "department",
    "salary",
    "city"
)

selected_employee_df.show()

+-------+----------+------+---------+
|   name|department|salary|     city|
+-------+----------+------+---------+
|Kalyani|        IT| 50000|Hyderabad|
|  Rahul|        IT| 65000|Bangalore|
| Ramani|   Finance| 55000|Hyderabad|
|  Kiran|        IT| 70000|Bangalore|
| Suresh|   Finance| 60000|  Chennai|
|  Priya|        IT| 52000|Hyderabad|
+-------+----------+------+---------+



In [ ]:
transformed_employee_df = selected_employee_df.withColumn(
    "salary_category",
    F.when(F.col("salary") >= 65000, "High")
     .when(F.col("salary") >= 50000, "Medium")
     .otherwise("Low")
)

transformed_employee_df.show()

+-------+----------+------+---------+---------------+
|   name|department|salary|     city|salary_category|
+-------+----------+------+---------+---------------+
|Kalyani|        IT| 50000|Hyderabad|         Medium|
|  Rahul|        IT| 65000|Bangalore|           High|
| Ramani|   Finance| 55000|Hyderabad|         Medium|
|  Kiran|        IT| 70000|Bangalore|           High|
| Suresh|   Finance| 60000|  Chennai|         Medium|
|  Priya|        IT| 52000|Hyderabad|         Medium|
+-------+----------+------+---------+---------------+



In [ ]:
department_data = [
    ("IT", "Suresh"),
    ("HR", "Priya"),
    ("Finance", "Arjun"),
    ("Sales", "Meena")
]

department_df = spark.createDataFrame(
    department_data,
    ["department", "manager"]
)

department_df.show()

+----------+-------+
|department|manager|
+----------+-------+
|        IT| Suresh|
|        HR|  Priya|
|   Finance|  Arjun|
|     Sales|  Meena|
+----------+-------+



In [ ]:
joined_employee_df = transformed_employee_df.join(
    department_df,
    on="department",
    how="left"
)

joined_employee_df.show()

+----------+-------+------+---------+---------------+-------+
|department|   name|salary|     city|salary_category|manager|
+----------+-------+------+---------+---------------+-------+
|   Finance| Ramani| 55000|Hyderabad|         Medium|  Arjun|
|   Finance| Suresh| 60000|  Chennai|         Medium|  Arjun|
|        IT|Kalyani| 50000|Hyderabad|         Medium| Suresh|
|        IT|  Rahul| 65000|Bangalore|           High| Suresh|
|        IT|  Kiran| 70000|Bangalore|           High| Suresh|
|        IT|  Priya| 52000|Hyderabad|         Medium| Suresh|
+----------+-------+------+---------+---------------+-------+



In [ ]:
final_employee_df = joined_employee_df.select(
    "name",
    "department",
    "manager",
    "salary",
    "salary_category",
    "city"
)

final_employee_df.show()

+-------+----------+-------+------+---------------+---------+
|   name|department|manager|salary|salary_category|     city|
+-------+----------+-------+------+---------------+---------+
|Kalyani|        IT| Suresh| 50000|         Medium|Hyderabad|
|  Rahul|        IT| Suresh| 65000|           High|Bangalore|
| Ramani|   Finance|  Arjun| 55000|         Medium|Hyderabad|
|  Kiran|        IT| Suresh| 70000|           High|Bangalore|
| Suresh|   Finance|  Arjun| 60000|         Medium|  Chennai|
|  Priya|        IT| Suresh| 52000|         Medium|Hyderabad|
+-------+----------+-------+------+---------------+---------+



In [ ]:
employee_count_df = final_employee_df.groupBy(
    "department"
).agg(
    F.count("*").alias("employee_count")
)

employee_count_df.show()

+----------+--------------+
|department|employee_count|
+----------+--------------+
|   Finance|             2|
|        IT|             4|
+----------+--------------+



In [ ]:
department_summary_df = final_employee_df.groupBy(
    "department"
).agg(
    F.count("*").alias("employee_count"),
    F.avg("salary").alias("average_salary"),
    F.max("salary").alias("maximum_salary")
)

department_summary_df.show()

+----------+--------------+--------------+--------------+
|department|employee_count|average_salary|maximum_salary|
+----------+--------------+--------------+--------------+
|   Finance|             2|       57500.0|         60000|
|        IT|             4|       59250.0|         70000|
+----------+--------------+--------------+--------------+



In [ ]:
department_summary_df = department_summary_df.orderBy(
    F.col("average_salary").desc()
)

department_summary_df.show()

+----------+--------------+--------------+--------------+
|department|employee_count|average_salary|maximum_salary|
+----------+--------------+--------------+--------------+
|        IT|             4|       59250.0|         70000|
|   Finance|             2|       57500.0|         60000|
+----------+--------------+--------------+--------------+



In [ ]:
final_employee_df.createOrReplaceTempView(
    "employee_view"
)

In [ ]:
spark.sql("""
    SELECT
        department,
        COUNT(*) AS employee_count,
        AVG(salary) AS average_salary,
        MAX(salary) AS maximum_salary
    FROM employee_view
    GROUP BY department
    ORDER BY average_salary DESC
""").show()

+----------+--------------+--------------+--------------+
|department|employee_count|average_salary|maximum_salary|
+----------+--------------+--------------+--------------+
|        IT|             4|       59250.0|         70000|
|   Finance|             2|       57500.0|         60000|
+----------+--------------+--------------+--------------+



In [ ]:
final_employee_df.write \
    .mode("overwrite") \
    .parquet("/content/final_employee_data")

In [ ]:
saved_employee_df = spark.read.parquet(
    "/content/final_employee_data"
)

saved_employee_df.show()


+-------+----------+-------+------+---------------+---------+
|   name|department|manager|salary|salary_category|     city|
+-------+----------+-------+------+---------------+---------+
|Kalyani|        IT| Suresh| 50000|         Medium|Hyderabad|
|  Rahul|        IT| Suresh| 65000|           High|Bangalore|
| Ramani|   Finance|  Arjun| 55000|         Medium|Hyderabad|
|  Kiran|        IT| Suresh| 70000|           High|Bangalore|
| Suresh|   Finance|  Arjun| 60000|         Medium|  Chennai|
|  Priya|        IT| Suresh| 52000|         Medium|Hyderabad|
+-------+----------+-------+------+---------------+---------+

